In [1]:
%pip install numpy==2.1.3 pandas==2.2.3 scipy==1.16.3 scikit-learn==1.6.1 catboost==1.2.10 lightgbm==4.6.0 pyarrow==23.0.1 joblib==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 12.6 MB/s  0:00:07


In [2]:
from pathlib import Path
import zipfile, io
DATA_SOURCE = Path('data.zip')
RUN_DIR = Path('v3_final')
RUN_DIR.mkdir(parents=True, exist_ok=True)
FILENAMES = ['submission.csv', 'submission_alternative.csv']

In [3]:
import os, re
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from catboost import CatBoostClassifier
import lightgbm as lgb
import joblib
from IPython.display import display
CUTOFF = 60
SEED = 20261024
CV_SEEDS = [20261024, 20261041]
THREADS = min(8, os.cpu_count() or 2)
MIN_LOCAL_ROWS = 350
MIN_LOCAL_CLASS = 35
with zipfile.ZipFile(DATA_SOURCE) as z:
    root = (RUN_DIR / 'unpacked').resolve()
    z.extractall(root)
candidates = list(root.rglob('train.parquet'))
DATA_DIR = candidates[0].parent
print('Данные:', DATA_DIR.resolve(), '\nРезультаты:', RUN_DIR.resolve())

Данные: /content/v3_final/unpacked/data 
Результаты: /content/v3_final


In [4]:
FILES = ['train.parquet', 'test.parquet', 'courses.parquet', 'assessments.parquet', 'student_assessment.parquet', 'vle.parquet', 'student_vle.parquet', 'columns_description.csv', 'sample_submission.csv']
display(pd.read_csv(DATA_DIR / 'columns_description.csv').head(100))
T = {Path(f).stem: pd.read_parquet(DATA_DIR / f) for f in FILES if f.endswith('.parquet')}
for df in T.values():
    df.columns = df.columns.str.lower()
train, test, courses, assessments, sa, vle, sv = [T[k] for k in ['train', 'test', 'courses', 'assessments', 'student_assessment', 'vle', 'student_vle']]
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
E, K = ('enrollment_id', ['code_module', 'code_presentation'])
base = pd.concat([train.drop(columns='target'), test], ignore_index=True).set_index(E)
display(pd.DataFrame({k: {'rows': len(v), 'columns': len(v.columns)} for k, v in T.items()}).T)
display(train.groupby(K).target.agg(['count', 'mean']))

,table,column,description
0,train / test,enrollment_id,Ключ записи студента на поток курса. По нему с...
1,train / test,student_id,Ключ студента. Один студент может быть записан...
2,train / test,code_module,Код модуля (курса)
3,train / test,code_presentation,Код потока: год и буква B (старт в феврале) ил...
4,train / test,gender,Пол
5,train / test,region,Регион проживания
6,train / test,highest_education,Уровень образования на момент записи
7,train / test,imd_band,Индекс депривации района проживания (Index of ...
8,train / test,age_band,Возрастная группа
9,train / test,num_of_prev_attempts,Сколько раз студент уже пытался пройти этот мо...


,rows,columns
train,17566,14
test,8837,13
courses,22,3
assessments,206,6
student_assessment,51295,5
vle,6364,6
student_vle,4267890,4


count      mean
code_module code_presentation                 
AAA         2013J                363  0.234160
BBB         2013B               1455  0.448110
            2013J               1783  0.398766
            2014B               1248  0.417468
CCC         2014B               1352  0.509615
DDD         2013B               1077  0.526462
            2013J               1567  0.470964
            2014B                967  0.504654
EEE         2013J                904  0.326327
            2014B                586  0.390785
FFF         2013B               1395  0.439427
            2013J               1910  0.426702
            2014B               1250  0.476800
GGG         2013J                919  0.355822
            2014B                790  0.394937

In [5]:
def make_features(base, courses, assessments, sa, vle, sv):
    idx = base.index
    blocks = {}

    def put(name, frame, zero=False):
        frame = frame.reindex(idx)
        if zero:
            frame = frame.fillna(0)
        frame.columns = [name + '__' + str(c) for c in frame.columns]
        blocks[name] = frame
    allowed = ['code_module', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'date_registration']
    b = base[[c for c in allowed if c in base]].copy()
    b['season'] = base.code_presentation.str[-1]
    b['length'] = base.reset_index().merge(courses, on=K, validate='many_to_one').set_index(E).module_presentation_length
    b['remaining_course_days'] = b['length'] - CUTOFF
    b['fraction_course_elapsed'] = CUTOFF / b['length']
    put('base', b)
    events = sv.loc[sv.sum_click > 0].copy()
    daily = events.groupby([E, 'date'], observed=True).sum_click.sum().unstack(fill_value=0).reindex(idx, fill_value=0)
    arr = daily.reindex(columns=range(0, CUTOFF + 1), fill_value=0).to_numpy(dtype=np.float64)
    act = arr > 0
    g = events.groupby(E, observed=True)
    basic = g.agg(clicks=('sum_click', 'sum'), event_rows=('sum_click', 'size'), sites=('id_site', 'nunique'), active_days=('date', 'nunique'), first_day=('date', 'min'), last_day=('date', 'max')).reindex(idx)
    for c in ['clicks', 'event_rows', 'sites', 'active_days']:
        basic[c] = basic[c].fillna(0)
    basic['recency'] = CUTOFF - basic.last_day
    basic['span'] = basic.last_day - basic.first_day
    basic['clicks_per_active_day'] = basic.clicks / basic.active_days.replace(0, np.nan)
    basic['no_activity'] = (basic.clicks == 0).astype(int)
    nsites = vle.groupby(K).id_site.nunique()
    basic['available_sites'] = pd.MultiIndex.from_frame(base[K]).map(nsites).to_numpy()
    basic['site_coverage'] = basic.sites / basic.available_sites
    put('basic', basic)

    def windows_features(windows):
        out = {}
        for name, lo, hi in windows:
            cols = [d for d in daily.columns if lo <= d <= hi]
            a = daily[cols].to_numpy(dtype=float)
            out[name + '_clicks'] = a.sum(1)
            out[name + '_active'] = (a > 0).sum(1)
            out[name + '_rate'] = a.sum(1) / max(1, hi - lo + 1)
        return pd.DataFrame(out, index=idx)
    windows = [('d0_13', 0, 13), ('d14_27', 14, 27), ('d28_41', 28, 41), ('d42_60', 42, CUTOFF)]
    windows += [(f'last{n}', CUTOFF - n + 1, CUTOFF) for n in [3, 7, 14, 30]]
    windows += [(f'week{i}', 7 * i, min(7 * i + 6, CUTOFF)) for i in range(9)]
    w = windows_features(windows)
    w['last7_vs_prev7'] = (arr[:, -7:].sum(1) + 1) / (arr[:, -14:-7].sum(1) + 1)
    w['last14_share'] = (arr[:, -14:].sum(1) + 1) / (arr.sum(1) + 1)
    w['late_vs_early'] = (arr[:, 31:].sum(1) + 1) / (arr[:, :31].sum(1) + 1)
    put('windows', w)
    earliest = min(-1, int(events.date.min())) if len(events) else -1
    pre = windows_features([('all', earliest, -1), ('last7', -7, -1), ('last30', -30, -1)])
    pre['pre_share'] = pre.all_clicks / (basic.clicks + 1)
    put('pre', pre)
    shape = pd.DataFrame(index=idx)
    shape['zero_fraction'] = (~act).mean(1)
    shape['mean'] = arr.mean(1)
    shape['std'] = arr.std(1)
    shape['max'] = arr.max(1)
    shape['cv'] = arr.std(1) / (arr.mean(1) + 1)
    shape['burst_share'] = arr.max(1) / (arr.sum(1) + 1)
    x = np.arange(arr.shape[1])
    xc = x - x.mean()
    shape['slope_log'] = np.log1p(arr).dot(xc) / (xc @ xc)
    p = arr / np.maximum(arr.sum(1, keepdims=True), 1)
    shape['entropy'] = -(p * np.log(np.maximum(p, 1e-12))).sum(1)
    for half in [3, 7, 14, 30]:
        shape[f'decay{half}'] = arr.dot(2.0 ** (-(CUTOFF - x) / half))
    z = np.log1p(arr)
    for lag in [1, 7]:
        a, b = (z[:, :-lag], z[:, lag:])
        a = a - a.mean(1, keepdims=True)
        b = b - b.mean(1, keepdims=True)
        shape[f'autocorr{lag}'] = (a * b).sum(1) / np.maximum(np.sqrt((a * a).sum(1) * (b * b).sum(1)), 1e-12)

    def runs(row):
        pos = np.flatnonzero(row)
        bounds = np.r_[-1, pos, len(row)]
        zero_run = np.diff(bounds).max() - 1
        if not len(pos):
            return [zero_run, 0, np.nan, np.nan]
        chunks = np.split(pos, np.flatnonzero(np.diff(pos) > 1) + 1)
        gaps = np.diff(pos)
        return [zero_run, max(map(len, chunks)), gaps.mean() if len(gaps) else np.nan, gaps.max() if len(gaps) else np.nan]
    shape[['max_zero_run', 'max_active_run', 'gap_mean', 'gap_max']] = np.array([runs(row) for row in act])
    put('shape', shape)
    put('raw', pd.DataFrame(np.log1p(arr), index=idx, columns=[f'day{d}' for d in x]))
    ev = events.merge(base[K], left_on=E, right_index=True, validate='many_to_one')
    ev = ev.merge(vle, on=K + ['id_site'], how='left', validate='many_to_one', indicator=True)
    ev['activity_type'] = ev.activity_type.fillna('unknown').astype(str)
    type_parts = []
    for name, lo, hi in [('all', earliest, CUTOFF), ('pre', earliest, -1), ('recent', 47, CUTOFF), ('early', 0, 20)]:
        q = ev.loc[ev.date.between(lo, hi)]
        tab = q.groupby([E, 'activity_type'], observed=True).sum_click.sum().unstack(fill_value=0)
        tab.columns = [f'{name}_{t}' for t in tab.columns]
        type_parts.append(tab)
    ty = pd.concat(type_parts, axis=1).reindex(idx).fillna(0)
    allcols = [c for c in ty if c.startswith('all_')]
    ty['n_types'] = (ty[allcols] > 0).sum(1)
    for c in allcols:
        ty[c + '_share'] = ty[c] / (basic.clicks + 1)
    if 'week_from' in ev and 'week_to' in ev:
        ev['ahead_clicks'] = ev.sum_click * ((ev.week_from - 1) * 7 > ev.date)
        ev['behind_clicks'] = ev.sum_click * (ev.week_to * 7 - 1 < ev.date)
        ty = ty.join(ev.groupby(E)[['ahead_clicks', 'behind_clicks']].sum()).fillna(0)
    put('types', ty)
    sched = base.reset_index()[[E] + K].merge(assessments, on=K, how='left', validate='many_to_many')
    sched = sched.merge(sa, on=[E, 'id_assessment'], how='left', validate='one_to_one', indicator=True)
    sched['submitted'] = (sched['_merge'] == 'both').astype(int)
    sched['due'] = (sched.date.notna() & (sched.date <= CUTOFF)).astype(int)
    sched['missed'] = sched.due * (1 - sched.submitted)
    sched['late'] = (sched.submitted.eq(1) & sched.date.notna() & (sched.date_submitted > sched.date)).astype(int)
    sched['delay'] = sched.date_submitted - sched.date
    sched['score_known'] = (sched.submitted.eq(1) & sched.score.notna()).astype(int)
    sched['failed'] = (sched.score_known.eq(1) & (sched.score < 40)).astype(int)
    sched['weighted_score'] = sched.score * sched.weight / 100
    sched['due_weight'] = sched.weight * sched.due
    sched['missing_weight'] = sched.weight * sched.missed
    sched['banked_count'] = sched.is_banked.fillna(0) * sched.submitted
    ag = sched.groupby(E).agg(planned=('id_assessment', 'count'), due=('due', 'sum'), submitted=('submitted', 'sum'), missed=('missed', 'sum'), late=('late', 'sum'), known_scores=('score_known', 'sum'), failed=('failed', 'sum'), banked=('banked_count', 'sum'), score_mean=('score', 'mean'), score_min=('score', 'min'), score_max=('score', 'max'), score_std=('score', 'std'), delay_mean=('delay', 'mean'), delay_max=('delay', 'max'), last_submission=('date_submitted', 'max'), weighted_score=('weighted_score', 'sum'), due_weight=('due_weight', 'sum'), missing_weight=('missing_weight', 'sum'))
    ag['missed_ratio'] = ag.missed / ag.due.replace(0, np.nan)
    ag['score_missing'] = ag.submitted - ag.known_scores
    ag['submission_recency'] = CUTOFF - ag.last_submission
    ag['future_known'] = sched.loc[sched.date > CUTOFF].groupby(E).size()
    ag['next_deadline'] = sched.loc[sched.date > CUTOFF].groupby(E).date.min() - CUTOFF
    ag['recently_due'] = sched.loc[sched.date.between(CUTOFF - 14, CUTOFF)].groupby(E).size()
    for c in ['future_known', 'recently_due']:
        ag[c] = ag[c].fillna(0)
    for kind in sorted(assessments.assessment_type.dropna().unique()):
        q = sched.loc[sched.assessment_type == kind]
        z = q.groupby(E)[['submitted', 'missed', 'failed']].sum().add_prefix(str(kind) + '_')
        ag = ag.join(z.reindex(ag.index).fillna(0))
    q = sched.loc[sched.due.eq(1)].sort_values([E, 'date', 'id_assessment']).copy()
    q['order'] = q.groupby(E).cumcount()
    for n in range(4):
        a = q[q.order.eq(n)].set_index(E)[['submitted', 'score', 'delay', 'weight']]
        ag = ag.join(a.add_prefix(f'assignment{n}_'))
    put('assessment', ag)
    min_day = min(earliest, -30)
    axis = np.arange(min_day, CUTOFF + 1)
    mat = daily.reindex(columns=axis, fill_value=0).to_numpy(dtype=float)
    pref = np.pad(np.cumsum(mat, axis=1), ((0, 0), (1, 0)))
    pref_active = np.pad(np.cumsum(mat > 0, axis=1), ((0, 0), (1, 0)))
    dd = sched.loc[sched.date.notna() & (sched.date <= CUTOFF + 7), [E, 'date']].copy()
    rows = idx.get_indexer(dd[E])
    cols = []
    for label, low, high in [('before7', -7, -1), ('near3', -3, 3), ('after7', 1, 7)]:
        lo = np.maximum(min_day, dd.date.to_numpy().astype(int) + low)
        hi = np.minimum(CUTOFF, dd.date.to_numpy().astype(int) + high)
        valid = hi >= lo
        li = np.clip(lo - min_day, 0, len(axis))
        ri = np.clip(hi - min_day + 1, 0, len(axis))
        clicks = np.where(valid, pref[rows, ri] - pref[rows, li], 0)
        active = np.where(valid, pref_active[rows, ri] - pref_active[rows, li], 0)
        observed = np.maximum(hi - lo + 1, 0)
        dd[label + '_clicks'] = np.where(valid, clicks, np.nan)
        dd[label + '_rate'] = np.divide(clicks, observed, out=np.full(len(dd), np.nan), where=observed > 0)
        dd[label + '_active'] = np.where(valid, active, np.nan)
        dd[label + '_observed_fraction'] = observed / (high - low + 1)
        cols += [label + s for s in ['_clicks', '_rate', '_active', '_observed_fraction']]
    dl = dd.groupby(E)[cols].agg(['mean', 'max', 'min'])
    dl.columns = ['_'.join(c) for c in dl.columns]
    put('deadline', dl)
    features = pd.concat(blocks.values(), axis=1)
    numeric = features.select_dtypes(include='number').columns
    features[numeric] = features[numeric].replace([np.inf, -np.inf], np.nan).astype('float32')
    return (features, {k: list(v.columns) for k, v in blocks.items()})
F0, BLOCKS0 = make_features(base, courses, assessments, sa, vle, sv)
print('Базовые признаки:', F0.shape)

Базовые признаки: (26403, 350)


In [6]:
def make_extra_features(base, assessments, sa, vle, sv, F0):
    idx = base.index
    extra = {}

    def put(name, df):
        df = df.reindex(idx).copy()
        df.columns = [name + '__' + str(c) for c in df.columns]
        extra[name] = df
    schedule = assessments.sort_values(K + ['assessment_type', 'date', 'id_assessment'], na_position='last').copy()
    schedule['type_order'] = schedule.groupby(K + ['assessment_type']).cumcount()
    q = base.reset_index()[[E] + K].merge(schedule, on=K, validate='many_to_many')
    q = q.merge(sa, on=[E, 'id_assessment'], how='left', validate='one_to_one', indicator=True)
    q['submitted'] = q['_merge'].eq('both').astype(int)
    q['due'] = (q.date.notna() & (q.date <= CUTOFF)).astype(int)
    q['missing'] = q.due * (1 - q.submitted)
    q['known_score'] = (q.submitted.eq(1) & q.score.notna()).astype(int)
    q['late'] = (q.submitted.eq(1) & (q.date_submitted > q.date)).astype(int)
    q['early_future'] = (q.submitted.eq(1) & (q.date > CUTOFF)).astype(int)
    q['overdue_days'] = np.where(q.missing.eq(1), CUTOFF - q.date, 0)
    q['overdue_weight'] = q.overdue_days * q.weight.fillna(0)
    q['due_weight'] = q.weight.fillna(0) * q.due
    q['done_due_weight'] = q.due_weight * q.submitted
    q['known_weight'] = q.weight.fillna(0) * q.known_score
    q['score_weight'] = q.score.fillna(0) * q.known_weight
    q['on_time'] = (q.submitted.eq(1) & q.date.notna() & (q.date_submitted <= q.date)).astype(int)
    q['delay'] = q.date_submitted - q.date
    q['non_banked_score'] = q.score.where(q.is_banked.fillna(0).eq(0))
    for threshold in [40, 60, 80]:
        q[f'grade_ge{threshold}'] = (q.score >= threshold).astype(int)
    agg = q.groupby(E).agg(overdue_days_sum=('overdue_days', 'sum'), overdue_days_max=('overdue_days', 'max'), overdue_weight=('overdue_weight', 'sum'), due_weight=('due_weight', 'sum'), done_due_weight=('done_due_weight', 'sum'), known_weight=('known_weight', 'sum'), score_weight=('score_weight', 'sum'), early_future=('early_future', 'sum'), on_time=('on_time', 'sum'), known_score=('known_score', 'sum'), non_banked_score=('non_banked_score', 'mean'))
    agg['weighted_grade'] = agg.score_weight / agg.known_weight.replace(0, np.nan)
    agg['done_weight_fraction'] = agg.done_due_weight / agg.due_weight.replace(0, np.nan)
    agg['earned_per_due_weight'] = agg.score_weight / (100 * agg.due_weight.replace(0, np.nan))
    for t in [40, 60, 80]:
        agg[f'fraction_ge{t}'] = q.groupby(E)[f'grade_ge{t}'].sum() / agg.known_score.replace(0, np.nan)
    for kind, g in q.groupby('assessment_type', observed=True):
        stats = g.groupby(E)[['due', 'submitted', 'missing', 'late', 'on_time', 'known_score']].sum()
        stats['completion'] = 1 - stats['missing'] / stats['due'].replace(0, np.nan)
        stats['mean_score'] = g.groupby(E).score.mean()
        stats['min_score'] = g.groupby(E).score.min()
        agg = agg.join(stats.add_prefix(str(kind) + '_'))
    submitted = q[q.submitted.eq(1)].sort_values([E, 'date_submitted', 'id_assessment']).copy()
    rows = []
    for eid, g in submitted.groupby(E, sort=False):
        scores = g.score.dropna().to_numpy(dtype=float)
        valid = g[g.score.notna()]
        dates = valid.date_submitted.to_numpy(dtype=float)
        slope = np.nan
        if len(scores) > 1 and np.var(dates) > 0:
            slope = np.dot(dates - dates.mean(), scores - scores.mean()) / np.sum((dates - dates.mean()) ** 2)
        rows.append({E: eid, 'last_score': scores[-1] if len(scores) else np.nan, 'last2_mean': scores[-2:].mean() if len(scores) else np.nan, 'first_last_delta': scores[-1] - scores[0] if len(scores) > 1 else np.nan, 'grade_slope': slope, 'last_delay': g.delay.iloc[-1], 'latest_is_banked': g.is_banked.iloc[-1]})
    if rows:
        agg = agg.join(pd.DataFrame(rows).set_index(E))
    for kind in sorted(schedule.assessment_type.dropna().unique()):
        for n in range(8):
            s = q[q.assessment_type.eq(kind) & q.type_order.eq(n)].set_index(E)
            vals = s[['date', 'weight', 'due', 'submitted', 'missing', 'score', 'delay']].copy()
            vals['exists'] = 1
            agg = agg.join(vals.add_prefix(f'{kind}{n}_'))
    put('assessment2', agg)
    ev = sv[sv.sum_click > 0].merge(base[K], left_on=E, right_index=True, validate='many_to_one')
    ev = ev.merge(vle, on=K + ['id_site'], how='left', validate='many_to_one')
    ev['activity_type'] = ev.activity_type.fillna('unknown').astype(str)
    ar = []
    for kind, g in ev.groupby('activity_type', observed=True):
        a = g.groupby(E).agg(last=('date', 'max'), first=('date', 'min'), sites=('id_site', 'nunique'))
        a['recency'] = CUTOFF - a['last']
        for tag, lo, hi in [('pre', -100000, -1), ('start', 0, 13), ('middle', 14, 39), ('prev', 40, 53), ('last', 54, 60)]:
            gg = g[g.date.between(lo, hi)]
            a[tag + '_clicks'] = gg.groupby(E).sum_click.sum().reindex(a.index).fillna(0)
            a[tag + '_days'] = gg.groupby(E).date.nunique().reindex(a.index).fillna(0)
        a['recent_rate_ratio'] = (a.last_clicks / 7 + 1) / (a.prev_clicks / 14 + 1)
        ar.append(a.add_prefix(str(kind) + '_'))
    act = pd.concat(ar, axis=1) if ar else pd.DataFrame(index=idx)
    for k in range(8):
        g = ev[ev.date.between(CUTOFF - 7 * k - 6, CUTOFF - 7 * k)]
        w = g.groupby(E).agg(clicks=('sum_click', 'sum'), sites=('id_site', 'nunique'), days=('date', 'nunique'))
        act = act.join(w.add_prefix(f'backweek{k}_'))
    act = act.reindex(idx).copy()
    counts = [c for c in act if c.endswith(('_clicks', '_days', '_sites'))]
    act[counts] = act[counts].fillna(0)
    act['grade_x_recent'] = F0['assessment__score_mean'] * np.log1p(F0['windows__last14_clicks'])
    act['missed_x_recent'] = F0['assessment__missed_ratio'] * np.log1p(F0['windows__last14_clicks'])
    act['submission_vs_activity_recency'] = F0['assessment__submission_recency'] - F0['basic__recency']
    put('activity2', act)
    d = ev.groupby([E, 'date']).sum_click.sum().unstack(fill_value=0).reindex(idx, fill_value=0)
    min_day = min(-30, int(d.columns.min())) if len(d.columns) else -30
    axis = np.arange(min_day, CUTOFF + 1)
    arr = d.reindex(columns=axis, fill_value=0).to_numpy(float)
    pref = np.pad(np.cumsum(arr, axis=1), ((0, 0), (1, 0)))
    due = q[q.due.eq(1)].sort_values([E, 'date', 'id_assessment']).copy()
    row = idx.get_indexer(due[E])
    deadline = due.date.to_numpy(dtype=int)
    for name, lo, hi in [('before14', -14, -8), ('before7', -7, -1), ('near', -1, 1), ('after7', 1, 7)]:
        left = np.maximum(min_day, deadline + lo)
        right = np.minimum(CUTOFF, deadline + hi)
        n = np.maximum(right - left + 1, 0)
        li = np.clip(left - min_day, 0, len(axis))
        ri = np.clip(right - min_day + 1, 0, len(axis))
        val = np.where(n > 0, pref[row, ri] - pref[row, li], 0)
        due[name + '_rate'] = np.divide(val, n, out=np.full(len(due), np.nan), where=n > 0)
        due[name + '_observed'] = n / (hi - lo + 1)
    due['acceleration'] = (due.before7_rate + 1) / (due.before14_rate + 1)
    due['after_vs_before'] = (due.after7_rate + 1) / (due.before7_rate + 1)
    cols = [c for c in due if c.endswith(('_rate', '_observed'))] + ['acceleration', 'after_vs_before']
    dl = due.groupby(E)[cols].agg(['mean', 'max'])
    dl.columns = ['_'.join(c) for c in dl.columns]
    for label, group in [('last', due.groupby(E).tail(1)), ('first', due.groupby(E).head(1))]:
        dl = dl.join(group.set_index(E)[cols + ['date', 'missing', 'score']].add_prefix(label + '_'))
    missed = due[due.missing.eq(1)].groupby(E)[['after7_rate', 'before7_rate']].mean().add_prefix('missed_')
    dl = dl.join(missed)
    put('deadline2', dl)
    cb = pd.concat([F0, extra['assessment2']], axis=1)
    norm_cols = ['basic__clicks', 'basic__active_days', 'basic__recency', 'basic__sites', 'windows__last7_clicks', 'windows__last14_clicks', 'pre__all_clicks', 'assessment__score_mean', 'assessment__missed_ratio', 'assessment__weighted_score', 'assessment2__weighted_grade', 'assessment2__done_weight_fraction']
    cohort = pd.DataFrame(index=idx)
    keys = [base[c] for c in K]
    for c in norm_cols:
        s = cb[c]
        gg = s.groupby(keys, dropna=False)
        cohort[c + '_rank'] = gg.rank(method='average', pct=True)
        mu = gg.transform('mean')
        std = gg.transform('std').replace(0, np.nan)
        cohort[c + '_z'] = (s - mu) / std
        cohort[c + '_relative'] = s / (mu.abs() + 1)
    scored = q[q.known_score.eq(1)].copy()
    sg = scored.groupby('id_assessment').score
    scored['score_rank'] = sg.rank(pct=True)
    scored['score_z'] = (scored.score - sg.transform('mean')) / sg.transform('std').replace(0, np.nan)
    ss = scored.groupby(E)[['score_rank', 'score_z']].agg(['mean', 'min', 'max'])
    ss.columns = ['_'.join(c) for c in ss.columns]
    cohort = cohort.join(ss)
    put('cohort', cohort)
    hist_cols = ['basic__clicks', 'basic__active_days', 'basic__recency', 'assessment__score_mean', 'assessment__missed_ratio', 'windows__last14_clicks']
    period = base.code_presentation.map({'2013B': 0, '2013J': 1, '2014B': 2, '2014J': 3})
    history = pd.DataFrame(index=idx)
    for p in sorted(period.unique()):
        current = idx[period.eq(p)]
        previous = idx[period.lt(p)]
        past = F0.loc[previous, hist_cols].join(base.loc[previous, ['student_id', 'code_module']])
        stats = past.groupby('student_id')[hist_cols].agg(['mean', 'max'])
        stats.columns = ['_'.join(c) for c in stats.columns]
        stats['past_enrollments'] = past.groupby('student_id').size()
        stats['past_modules'] = past.groupby('student_id').code_module.nunique()
        mapped = stats.reindex(base.loc[current, 'student_id'].values)
        mapped.index = current
        for c in mapped:
            history.loc[current, c] = mapped[c]
        for c in hist_cols:
            history.loc[current, c + '_vs_history'] = F0.loc[current, c] - history.loc[current, c + '_mean']
    history['past_enrollments'] = history.past_enrollments.fillna(0)
    history['past_modules'] = history.past_modules.fillna(0)
    put('history', history)
    new = pd.concat(extra.values(), axis=1)
    new = new.replace([np.inf, -np.inf], np.nan).astype('float32')
    return (new, {k: list(v.columns) for k, v in extra.items()})
Fnew, BLOCKSnew = make_extra_features(base, assessments, sa, vle, sv, F0)
F = pd.concat([F0, Fnew], axis=1)
BLOCKS = {**BLOCKS0, **BLOCKSnew}
print('Всего признаков:', F.shape)
display(pd.Series({k: len(v) for k, v in BLOCKS.items()}, name='columns'))

/tmp/ipykernel_1074/3401511676.py:82: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  act['grade_x_recent'] = F0['assessment__score_mean'] * np.log1p(F0['windows__last14_clicks'])
/tmp/ipykernel_1074/3401511676.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  act['missed_x_recent'] = F0['assessment__missed_ratio'] * np.log1p(F0['windows__last14_clicks'])
/tmp/ipykernel_1074/3401511676.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perf

Всего признаков: (26403, 1014)


,columns
base,14
basic,12
windows,54
pre,10
shape,18
raw,61
types,96
assessment,49
deadline,36
assessment2,242


In [7]:
CORE = ['base', 'basic', 'windows', 'pre', 'shape', 'types', 'assessment', 'deadline']
A = CORE + ['assessment2']
B = A + ['activity2', 'deadline2']
C = B + ['cohort']
D = C + ['history']
FULL = D + ['raw']
DEEP = dict(depth=7, l2_leaf_reg=15, random_strength=1.5)

def recipe(name, blocks, weight='none', alphas=(0.0,)):
    return dict(name=name, blocks=blocks, kind='cb', params=DEEP, weight=weight, alphas=list(alphas), checks=[500], portable=False)
SPECS = [recipe('full_specialists', FULL, alphas=(0.25,)), recipe('full_Jweight', FULL, weight='J'), recipe('behavior_v2', B), recipe('cohort_local', C, alphas=(0.25,))]
BY_NAME = {s['name']: s for s in SPECS}
y = train.set_index(E).target.astype(int)
train_ids = train[E].to_numpy()
test_ids = test[E].to_numpy()

In [8]:
def prepare_frames(tr, va, columns, kind):
    a = F.loc[tr, columns].copy()
    b = F.loc[va, columns].copy()
    keep = [c for c in a if a[c].nunique(dropna=False) > 1]
    a = a[keep]
    b = b[keep]
    cats = list(a.select_dtypes(include=['object', 'string', 'category']).columns)
    for c in cats:
        aa = a[c].astype('string').fillna('__MISSING__').astype(str)
        bb = b[c].astype('string').fillna('__MISSING__').astype(str)
        if kind == 'cb':
            a[c], b[c] = (aa, bb)
        else:
            levels = sorted(aa.unique())
            a[c] = pd.Categorical(aa, categories=levels)
            b[c] = pd.Categorical(bb, categories=levels)
    return (a, b, cats)

def train_weights(tr, mode):
    w = np.ones(len(tr))
    if mode == 'J':
        w = np.where(base.loc[tr, 'code_presentation'].str.endswith('J'), 1.5, 1.0)
    return w / w.mean()

def model_factory(s, trees, seed, local=False):
    params = dict(s['params'])
    if s['kind'] == 'cb':
        if local:
            params = dict(depth=4, l2_leaf_reg=25, random_strength=1.0)
        return CatBoostClassifier(iterations=int(trees), learning_rate=0.035, loss_function='Logloss', eval_metric='AUC', random_seed=seed, thread_count=THREADS, verbose=False, allow_writing_files=False, **params)
    if local:
        params = dict(num_leaves=7, min_child_samples=30, reg_lambda=20)
    return lgb.LGBMClassifier(n_estimators=int(trees), learning_rate=0.035, objective='binary', metric='auc', random_state=seed, n_jobs=THREADS, verbosity=-1, deterministic=True, force_col_wise=True, **params)

def fit_one(s, tr, va, seed, checks, local=False):
    columns = [c for b in s['blocks'] for c in BLOCKS[b]]
    if s.get('portable', False):
        columns = [c for c in columns if not re.search(PORTABLE_DROP, c)]
    a, b, cats = prepare_frames(tr, va, columns, s['kind'])
    model = model_factory(s, max(checks), seed, local=local)
    kwargs = {'cat_features': cats, 'verbose': False} if s['kind'] == 'cb' else {}
    model.fit(a, y.loc[tr], sample_weight=train_weights(tr, s['weight']), **kwargs)
    predictions = {}
    for t in checks:
        kw = {'ntree_end': int(t)} if s['kind'] == 'cb' else {'num_iteration': int(t)}
        predictions[t] = model.predict_proba(b, **kw)[:, 1]
    artifact = {'model': model, 'kind': s['kind'], 'columns': list(a.columns), 'categories': {c: list(a[c].cat.categories) for c in cats if s['kind'] == 'lgb'}}
    return (predictions, artifact)

def variant_key(name, trees, alpha):
    return f'{name}@{trees}:a{alpha:g}'

def pipeline_arrays(s, tr, va, seed, checks, alphas, save_prefix=None):
    gp, artifact = fit_one(s, tr, va, seed, checks)
    if save_prefix:
        joblib.dump(artifact, str(save_prefix) + '_global.joblib')
    lp = {t: p.copy() for t, p in gp.items()}
    if max(alphas) > 0:
        for module in sorted(base.loc[va, 'code_module'].unique()):
            tt = tr[base.loc[tr, 'code_module'].eq(module)]
            pos = np.flatnonzero(base.loc[va, 'code_module'].eq(module).to_numpy())
            counts = y.loc[tt].value_counts()
            if len(tt) < MIN_LOCAL_ROWS or len(counts) < 2 or counts.min() < MIN_LOCAL_CLASS:
                continue
            pp, art = fit_one(s, tt, va[pos], seed, checks, local=True)
            for t in checks:
                lp[t][pos] = pp[t]
            if save_prefix:
                joblib.dump(art, str(save_prefix) + '_local_' + str(module) + '.joblib')
    return {variant_key(s['name'], t, a): (1 - a) * gp[t] + a * lp[t] for t in checks for a in alphas}

In [9]:
parts = {}
for name in ['behavior_v2', 'full_specialists', 'full_Jweight', 'cohort_local']:
    spec = BY_NAME[name]
    parts[name] = {}
    for seed in CV_SEEDS:
        predictions = pipeline_arrays(spec, train_ids, test_ids, seed, [500], spec['alphas'], save_prefix=None)
        parts[name][seed] = predictions[variant_key(name, 500, spec['alphas'][0])]
    print('Готово:', name, 'seed', CV_SEEDS, flush=True)

Готово: behavior_v2 seed [20261024, 20261041]
Готово: full_specialists seed [20261024, 20261041]
Готово: full_Jweight seed [20261024, 20261041]
Готово: cohort_local seed [20261024, 20261041]


In [10]:
members = ['full_specialists', 'full_Jweight', 'behavior_v2']
bag = {name: np.mean([parts[name][seed] for seed in CV_SEEDS], axis=0) for name in parts}
main = np.average(np.stack([rankdata(bag[name], method='average') / len(test_ids) for name in members]), axis=0, weights=[1, 1, 1])
reference_raw = np.average(np.stack([rankdata(parts[name][SEED], method='average') / len(test_ids) for name in members]), axis=0, weights=[1, 1, 1])
reference_frame = pd.DataFrame({'id': sample.id, 'prediction': sample.id.map(pd.Series(reference_raw, index=test_ids))})
reference_csv = reference_frame.to_csv(index=False, float_format='%.10f')
reference = pd.read_csv(io.StringIO(reference_csv)).set_index('id').prediction.reindex(test_ids).to_numpy()
new_rank = np.average(np.stack([rankdata(bag['cohort_local'], method='average') / len(test_ids)]), axis=0, weights=[1])
alternative = np.average(np.stack([reference, new_rank]), axis=0, weights=[0.75, 0.25])
for filename, prediction in zip(FILENAMES, [main, alternative]):
    output = pd.DataFrame({'id': sample.id, 'prediction': sample.id.map(pd.Series(prediction, index=test_ids))})
    output.to_csv(RUN_DIR / filename, index=False, float_format='%.10f')